# Road Surface Defect Detection — Robustness & Adversarial Evaluation
## Детекция дефектов дорожного покрытия: устойчивость и состязательные атаки

*ML course project — end-to-end: data engineering → training → attacks → defense → evaluation → deployment, plus a red-team attack on another team's model.*

**О чём проект.** Обучить детектор дефектов дороги (**выбоина**, **трещина**), **честно** оценить его на реальных
искажениях и целенаправленных атаках и повысить устойчивость, не потеряв точность на чистых кадрах.
Вторая часть — red-team: **атака на чужую модель** (детектор ножа другой команды).

> **Формат.** Это портфолио-разбор: ключевые куски **реального кода** проекта + **настоящие** графики и метрики из
> проведённых прогонов. Ноутбук не переобучает модели при запуске (дедуп 10k кадров и 4 обучения YOLO — это часы GPU);
> все числа ниже получены в реальных экспериментах.

---

### TL;DR

| | |
|---|---|
| **Данные** | RDD2022 → 10 000 кадров → 960×960, 2 класса → DINOv2-дедуп (−~2000) → очистка боксов (−27) → +негативы |
| **Модель** | YOLO11s @ 960, AdamW, cosine LR; YOLO11m проверена для сравнения размера |
| **Качество** | test **mAP50 ≈ 0.51**; m не обошла s → упираемся в данные, не в размер сети |
| **Устойчивость** | погода/заклейка — терпимо (−11…−15 п. mAP); **PGD ломает модель полностью** (mAP→0) |
| **Защита** | негативы: ложные срабатывания 24% → 13%; дообучение на искажениях: погода ASR 43% → 34% |
| **Деплой** | Docker + FastAPI + ngrok (CPU-инференс, REST `/predict`) |
| **Red-team** | чужой детектор ножа обходится физически (цвет лезвия/скрытие рукояти); цифровая boundary-атака (HSJA) находит границу за ~20 запросов, но незаметный adversarial — дорого по бюджету запросов |

**Стек:** Python, PyTorch, Ultralytics YOLO11, OpenCV, DINOv2, SAM, NumPy/Pandas, FastAPI, Docker, ngrok.


---
## 1. Problem / Задача

*Two defect classes, honest evaluation under distortions and white-/black-box attacks, then hardening.*

- Находить на дорожных кадрах **выбоины** и **трещины**.
- Оценить устойчивость к реальным искажениям (погода, перекрытие объекта) и **состязательным атакам** (PGD).
- Повысить устойчивость, сохранив точность на чистых кадрах.
- Red-team: добиться, чтобы **чужой** детектор ножа переставал видеть нож.


---
## 2. Dataset / Данные

*RDD2022 → 10k frames → 960×960, 2 classes, semantic dedup, label cleaning, mined negatives.*

![Состав датасетов](figures/datasets.png)

- **Источник — RDD2022** (Road Damage Detection, 7 стран). Отобрано **10 000 кадров**, приведены к **960×960**,
  классы сведены к двум.
- **Дедупликация через DINOv2** (косинусная близость, порог **0.92**): удалено **~2000** почти одинаковых кадров
  (соседние кадры видео, повторные проезды) → ~8000 чистых.
- **Очистка разметки**: удалено 27 проблемных боксов (нулевая площадь, дубликаты, два класса на одном объекте, вложенные).
- **Негативы** (кадры без дефектов) учат модель не срабатывать на пустой дороге: **700 из RDD** (вне исходных 10 000)
  + **~140 бразильских** кадров (по одному на место).

| набор | кадров | без дефектов | назначение |
|---|---|---|---|
| train | 5724 | 1458 (25.5%) | обучение (с негативами) |
| val | 1046 | 105 | выбор лучшей эпохи |
| test | 1047 | 90 | итоговая оценка |

**Спец-тесты:** `neg` (300, ложные срабатывания), `weather` (419), `patch` (222), `pgd` (209), `maps` (196, сдвиг домена).

![Примеры основного набора](figures/samples_main.png)


### 2.1. Сведение классов и очистка разметки / Class mapping & label cleaning

*Faithful excerpt from `build_rdd2022_dataset.py` — RDD codes → 2 classes + the box-cleaning rules.*

In [ ]:
# Классы RDD2022 -> 2 наших класса  (из build_rdd2022_dataset.py)
NAMES = {0: "pothole / выбоина", 1: "crack / трещина"}
CLASS_MAP = {"D40": 0, "D00": 1, "D10": 1, "D20": 1, "Block crack": 1, "D0w0": 1}
# D01, D11 (швы), D43, D44 (стёртая разметка), D50 (люки), Repair -> не дефекты -> отбрасываются.
# Кадр, где после фильтра не осталось боксов, становится негативом (пустой .txt).

# Правила очистки разметки (одобрены на dataset_960):
IOU_DUP     = 0.8   # два бокса одного класса с IoU >= 0.8  -> оставить один
CONTAIN     = 0.9   # вложенный бокс: >= 90% его площади внутри внешнего
COVER_OUTER = 0.5   # внешний бокс удаляется, если вложенные покрывают >= 50% его площади
# pothole и crack на одном месте (IoU >= 0.8) -> оставить pothole; бокс нулевой площади -> удалить

def iou(a, b):
    iw = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    ih = max(0.0, min(a[4], b[4]) - max(a[2], b[2]))
    inter = iw * ih
    union = (a[3]-a[1])*(a[4]-a[2]) + (b[3]-b[1])*(b[4]-b[2]) - inter
    return inter / union if union > 0 else 0.0

### 2.2. Семантическая дедупликация (DINOv2) / Semantic dedup

*Near-duplicate removal with DINOv2 embeddings + cosine 0.92 — catches what pixel hashes miss.*

> Код ниже иллюстрирует **метод** (реконструкция канонического DINOv2-дедупа); в проекте этот шаг выполнялся отдельно.

In [ ]:
# DINOv2-эмбеддинги -> косинусная близость -> дубли выше порога 0.92
import torch, torch.nn.functional as F

dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").eval().cuda()

@torch.no_grad()
def embed(batch):                 # batch: (N,3,H,W), H,W кратны 14
    feats = dino(batch.cuda())    # (N, 384) — CLS-эмбеддинг на кадр
    return F.normalize(feats, dim=1)

emb = embed(all_frames)           # по всем ~10 000 кадрам (мини-батчами)
sim = emb @ emb.T                 # косинус = скалярное произведение нормированных векторов
dup = (sim > 0.92).triu(1)        # пары выше порога -> почти-дубли
# из каждой группы дублей оставляем один кадр  ->  ~10 000 - ~2000 = ~8000

### 2.3. Аудит утечек и копий (pHash + md5) / Leakage & duplicate audit

*Real code from `audit_rdd2022_dups.py`: perceptual hash guards against train↔val/test leakage.*

In [ ]:
import numpy as np
# pHash (perceptual hash) на основе DCT — из audit_rdd2022_dups.py
def _dct_mat(n):
    k = np.arange(n)[:, None]
    m = np.cos(np.pi * (2*np.arange(n)[None, :] + 1) * k / (2*n)) * np.sqrt(2/n)
    m[0] /= np.sqrt(2)
    return m
D32 = _dct_mat(32)

def phash_of(g):                              # g: серая миниатюра 32x32
    low = (D32 @ g.astype(np.float64) @ D32.T)[:8, :8].ravel()
    bits = low > np.median(low[1:])
    return int(np.packbits(bits).view(">u8")[0])

# расстояние Хэмминга между pHash:  <=4 -> почти наверняка один и тот же кадр.
# Кадр-кандидат в train, похожий (<=8) на кадр val/test/neg_test -> УТЕЧКА -> исключается.
# Дополнительно md5 ловит точные копии файлов.

### 2.4. Майнинг негативов (ORB + геометрия) / Negative mining

*From `add_negatives.py`: ORB keypoints + RANSAC reject multiple shots of the same place; pHash rejects leakage.*

In [ ]:
import cv2, numpy as np
# "Снято одно и то же место" -> совпадают характерные точки ORB, согласованные одной геометрией (RANSAC)
def orb_inliers(fa, fb):
    bf = cv2.BFMatcher(cv2.NORM_HAMMING); best = 0
    for (pa, da), (pb, db) in ((fa, fb), (fb, fa)):
        if da is None or db is None or len(da) < 8 or len(db) < 8:
            continue
        good = [m for m, m2 in (p for p in bf.knnMatch(da, db, k=2) if len(p) == 2)
                if m.distance < 0.75 * m2.distance]
        if len(good) < 8:
            continue
        Fm, mask = cv2.findFundamentalMat(pa[[m.queryIdx for m in good]],
                                          pb[[m.trainIdx for m in good]], cv2.FM_RANSAC, 1.0, 0.99)
        if mask is not None:
            best = max(best, int(mask.sum()))
    return best
# >= 15 согласованных точек -> то же место; из каждого места оставляем первый кадр по имени.
# Негатив также отклоняется, если pHash <= 8 к кадру из наборов для оценки (защита от утечки).

---
## 3. Pipeline / Пайплайн

*End-to-end flow from raw RDD2022 to evaluated, hardened, deployed model.*

![Пайплайн](figures/pipeline.png)

RDD2022 (10 000) → 960×960 + 2 класса → **DINOv2-дедуп** (0.92, −~2000) → **очистка разметки** (−27) →
**4 обучения** (базовая YOLO11s; YOLO11m для сравнения размера; основная YOLO11s +негативы; дообучение на искажениях) →
генерация искажений/атак (weather / patch / PGD) → **оценка** (mAP, P, R, ложные срабатывания) → **деплой**.


---
## 4. Training / Обучение

*YOLO11s vs YOLO11m, 960 px, AdamW, cosine LR. Same `val` protocol on the held-out test for all runs.*

Обучались **YOLO11s** и (для сравнения размера) **YOLO11m**, Ultralytics 8.4, 960 px, AdamW, косинусный LR, GPU RTX 3060.
Итоговой выбрана **s-версия**: m не превзошла s и вышла на плато.

In [ ]:
from ultralytics import YOLO

# Обучение (из road_damage_960_yolo.ipynb) — базовая и основная модели
YOLO("yolo11s.pt").train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=960, batch=BATCH, device=0, workers=WORKERS,
    seed=SEED, deterministic=False, patience=PATIENCE,
    optimizer="AdamW", lr0=1e-3, cos_lr=True, close_mosaic=10,   # мозаику выключаем за 10 эпох до конца
)

# Единообразная оценка на отложенном test (тот же вызов для всех 4 прогонов):
YOLO(BEST).val(data=str(DATA_YAML), split="test", imgsz=960, batch=BATCH, device=0, plots=True)

### Кривые обучения основной модели / Training curves

![Кривые](figures/curves.png)

Уже у s виден разрыв train/val (train-loss заметно ниже val-loss) — модель начинает запоминать обучающие кадры,
а val-метрика стоит на ~0.55. Это и есть сигнал «упираемся в данные».


---
## 5. Attacks & distortions / Атаки и искажения

*Three stressors on one frame: synthetic weather, SAM-masked patch occlusion, and white-box PGD.*

![Искажения](figures/distortions.png)

- **weather** — синтетическая погода (дождь/снег/метель) поверх кадра. На примере дождь гасит все детекции (3 → 0).
- **patch** — заклейка одного дефекта «донорским» асфальтом по маске **SAM** (box-промпт → маска формы дефекта →
  ~50% маски заполняется чистым асфальтом). Реалистичная имитация перекрытия:

![Атака patch крупным планом](figures/patch_demo.png)

- **PGD** — состязательная атака «белого ящика»: незаметный L∞-шум (ε = 8/255, α = 2/255, 10 итераций),
  максимизирующий YOLO-лосс. Кадр для глаза не меняется, но детектор ломается (на примере 3 → 66 ложных боксов).


### PGD — ядро атаки / core loop

*Real `pgd_batch` from `apply_pgd.py`: maximize the model's own YOLO loss within an L∞ ball.*

In [ ]:
import torch
EPS, ALPHA, STEPS = 8/255, 2/255, 10          # бюджет L∞, шаг, число итераций

def pgd_batch(x0, labs):                        # x0: чистые кадры (N,3,H,W) в [0,1]
    H, W = x0.shape[-2:]; _, r, left, top = letterbox_t(x0[:1])
    batch = make_targets(labs, H, W, r, left, top, x0.device)
    x = (x0 + torch.empty_like(x0).uniform_(-EPS, EPS)).clamp(0, 1)   # старт из случайной точки ε-шара
    for s in range(STEPS):
        x.requires_grad_(True)
        loss, _ = crit(net(letterbox_t(x)[0]), batch)                 # YOLO-лосс (box + cls + dfl)
        g, = torch.autograd.grad(loss.sum(), x)
        with torch.no_grad():
            x = x + ALPHA * g.sign()                                   # шаг по знаку градиента
            x = torch.min(torch.max(x, x0 - EPS), x0 + EPS).clamp(0, 1)  # проекция в ε-шар и в [0,1]
    return x.detach()

---
## 6. Defense / Защита (baseline)

*Two cheap, honest defenses — negatives against false positives, static adversarial fine-tuning against distortions.*

### 6.1. Негативы против ложных срабатываний
На 300 негативах, которых не видела ни одна модель (порог уверенности 0.25):

| модель | кадров с ложным срабатыванием | ложных боксов / кадр |
|---|---|---|
| базовая (без негативов) | 24.0% | 0.297 |
| **с негативами (основная)** | **13.3%** | **0.147** |

Ложные срабатывания сократились почти вдвое. Цена — −2.8 п. mAP50 на чистом test (0.541 → 0.513).

### 6.2. Дообучение на искажениях
Основная модель **дообучается** на смеси чистых и аугментированных кадров (weather + patch + PGD, train-часть),
от готовых весов с низким LR (1e-4), 20 эпох.

| набор (аугм.) | mAP50: основная → дообученная | потеряно объектов (ASR): основная → дообученная |
|---|---|---|
| weather | 0.349 → **0.435** | 43% → **34%** |
| patch | 0.416 → 0.410 | 30% → 31% |
| PGD | 0.000 → 0.000 | 100% → 99% |

**Итог:** погода — заметный выигрыш; заклейка — без эффекта (объект физически скрыт); PGD — без изменений
(статичная adversarial-тренировка ограничена: атаку надо пересчитывать каждую эпоху против текущих весов).

In [ ]:
# Дообучение на искажениях: от весов основной модели, LR в 10x ниже -> это дообучение, не обучение с нуля
YOLO(BEST_MAIN).train(
    data=str(DATA_YAML_AUG),          # train = чистые + weather + patch + PGD (train-часть)
    epochs=20, imgsz=960, batch=BATCH, device=0,
    optimizer="AdamW", lr0=1e-4, cos_lr=True,
)

---
## 7. Results / Результаты

*All four runs evaluated with the same `val` call on the same 1047-image test set.*

### 7.1. Четыре обучения на одном test

| обучение | модель | эпохи | mAP50 | mAP50-95 | P | R |
|---|---|---|---|---|---|---|
| базовая (без негативов) | yolo11s | 100 | **0.541** | 0.241 | 0.624 | 0.493 |
| сравнение размера | yolo11m | 70 (плато) | 0.517 | 0.232 | 0.587 | 0.472 |
| основная (+негативы) | yolo11s | 70 | 0.513 | 0.235 | 0.584 | 0.480 |
| дообучение на искажениях | yolo11s | 20 | 0.495 | 0.223 | 0.551 | 0.488 |

**YOLO11m (≈20.0M параметров, ~3× вычислений) не обошла YOLO11s (≈9.4M)** → ограничение в данных, а не в размере сети.

### 7.2. Сдвиг домена

| набор | mAP50 | mAP50-95 | P | R |
|---|---|---|---|---|
| test (исходный) | 0.513 | 0.235 | 0.584 | 0.480 |
| maps (Yandex Maps) | 0.218 | 0.096 | 0.316 | 0.279 |

На снимках Yandex Maps метрика вдвое ниже: другой ракурс/перспектива камеры + непривычные помехи (лужи, снег, трамвайные пути).

### 7.3. Устойчивость: до/после искажения (одни кадры, одна разметка)

![Устойчивость](figures/robustness.png)

| набор | mAP50 чистый → аугм. | Δ mAP50 | потеряно объектов (ASR) | ложные боксы чистый → аугм. |
|---|---|---|---|---|
| weather | 0.501 → 0.349 | −15.1 п. | **43%** | 277 → 165 |
| patch | 0.526 → 0.416 | −11.0 п. | **30%** | 142 → 132 |
| PGD | 0.568 → 0.000 | −56.8 п. | **100%** | 125 → **14451** |

Погода и заклейка — умеренная деградация; **PGD** полностью ломает модель (mAP → 0, лавина ложных боксов) — ожидаемо для белого ящика.


---
## 8. Deployment / Развёртывание

*Model wrapped as a CPU web service: Docker + FastAPI, exposed through an ngrok tunnel.*

- **Docker** — модель, код и зависимости в самодостаточном образе (инференс на **CPU**, GPU не нужен), запуск одной командой.
- **Веб-интерфейс + REST API** (FastAPI): страница загрузки кадра и endpoint `/predict`.
- **Доступ извне через ngrok** — у сервера «серый» IP (за NAT), туннель даёт публичный HTTPS-адрес для другой команды.

Ответ `/predict`: `detections` (класс, conf, бокс x1y1x2y2), `count`, `image` (с нарисованными боксами, base64), `speed_ms`.
Служебный `/health` — статус, размер входа, список классов.


---
## 9. Red team: attacking another team's knife detector / Атака на чужую модель

*Make a knife go undetected — physically (no weights) and digitally through the API (black-box boundary attack).*

### 9.1. Физические атаки (без доступа к весам)
Меняли сам нож в кадре (цвет, материал, перекрытие рукоятки) и переснимали.

![Физические атаки](figures/physical_attacks.png)

| приём | результат |
|---|---|
| розовое лезвие | нож **не детектится** ✓ |
| закрытая рукоять («перчатка») | нож **не детектится** ✓ |
| белая бумага вместо лезвия | детектится |
| красное лезвие | детектится |
| макс. яркость лезвия (блик) | детектится |

**Вывод:** детектор держится за **светлое металлически-серое лезвие характерной формы**, а не за «понятие ножа».
Смена цвета на неметаллический (розовый) или скрытие рукояти ломают детекцию.

### 9.2. Цифровая атака по границе (boundary / HSJA)
Чёрный ящик через API (ответ «нож есть/нет» + conf, без весов и градиентов) → **decision-based** атака **HopSkipJump**.

![Boundary-атака (HSJA)](figures/boundary_attack.png)

- **Границу нашли быстро:** старт со случайного шума → бинарный поиск к оригиналу находит точку, где нож
  **не детектится** (conf = 0.000), за **~20 запросов**.
- **Но точка далеко от оригинала:** сильный шум/затемнение (L2 ≈ 54 700, L∞ ≈ 86/255) — визуально уже не исходное фото.
- **Идти по границе дорого:** каждая оценка направления границы = десятки–сотни запросов; против сервиса с
  лимитом частоты (3 с/запрос) это очень затратно.

**Вывод:** границу решения реально найти дёшево, но получить **незаметное** adversarial-изображение, двигаясь
вдоль границы, — дорого. Главный барьер чёрно-ящичной атаки — **бюджет запросов**, а не сам факт уязвимости.


---
## 10. Conclusions / Выводы

- **Детектор обучен и честно оценён:** test mAP50 ≈ 0.51; на сдвиге домена — ожидаемо ниже.
- **Подготовка данных окупилась:** DINOv2-дедуп (0.92) убрал ~2000 почти-дублей + очистка боксов → меньше утечки похожих кадров между train/val/test.
- **Размер модели упёрся в данные:** YOLO11m не обошла YOLO11s и вышла на то же плато — брать модель крупнее s смысла нет.
- **Негативы** срезали ложные срабатывания почти вдвое (24% → 13%) ценой −2.8 п. mAP.
- **Устойчивость:** к погоде/заклейке модель терпима (теряет 30–43% объектов); к **PGD** без защиты уязвима полностью.
- **Дообучение на искажениях** подняло устойчивость к погоде (ASR 43% → 34%) ценой −1.8 п. на чистом test; к заклейке и PGD — без эффекта (статичный подход ограничен).
- **Red-team:** чужой детектор обходится физически (цвет/рукоять); цифровая boundary-атака находит границу дёшево (~20 запросов), но незаметный adversarial дорог по бюджету запросов.

---

*Stack: Python · PyTorch · Ultralytics YOLO11 · OpenCV · DINOv2 · SAM · NumPy/Pandas · FastAPI · Docker · ngrok.*
